<a href="https://colab.research.google.com/github/trpelka/osint-lead-pipeline/blob/main/osint_lead_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fastapi uvicorn pydantic httpx -q

import os, sqlite3, re, urllib.request
from typing import Optional
from fastapi import FastAPI, HTTPException, Query
from fastapi.middleware.cors import CORSMiddleware
from fastapi.testclient import TestClient
from pydantic import BaseModel, field_validator
from IPython.display import display, HTML

if os.path.exists("leads.db"):
    os.remove("leads.db")

app = FastAPI(title="OSINT Lead Pipeline")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

DB_PATH = "leads.db"

def get_db():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    conn = get_db()
    conn.execute("""CREATE TABLE IF NOT EXISTS leads (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        company TEXT NOT NULL, domain TEXT NOT NULL UNIQUE,
        email TEXT NOT NULL, notes TEXT)""")
    conn.commit()
    conn.close()

init_db()

class LeadCreate(BaseModel):
    company: str; domain: str; email: str; notes: Optional[str] = None
    @field_validator("email")
    @classmethod
    def val_email(cls, v):
        if not re.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", v): raise ValueError("bad email")
        return v

class LeadUpdate(BaseModel):
    company: Optional[str]=None; email: Optional[str]=None; notes: Optional[str]=None

class LeadOut(BaseModel):
    id: int; company: str; domain: str; email: str; notes: Optional[str]

@app.get("/api/leads", response_model=list[LeadOut])
def list_leads(search: Optional[str]=Query(None), skip: int=0, limit: int=20):
    conn = get_db()
    if search:
        q = f"%{search}%"
        rows = conn.execute("SELECT * FROM leads WHERE company LIKE ? OR domain LIKE ? LIMIT ? OFFSET ?", (q,q,limit,skip)).fetchall()
    else:
        rows = conn.execute("SELECT * FROM leads LIMIT ? OFFSET ?", (limit,skip)).fetchall()
    conn.close()
    return [dict(r) for r in rows]

@app.get("/api/leads/{lead_id}", response_model=LeadOut)
def get_lead(lead_id: int):
    conn = get_db()
    row = conn.execute("SELECT * FROM leads WHERE id=?", (lead_id,)).fetchone()
    conn.close()
    if not row: raise HTTPException(404, "Not found")
    return dict(row)

@app.post("/api/leads", response_model=LeadOut, status_code=201)
def create_lead(lead: LeadCreate):
    conn = get_db()
    try:
        cur = conn.execute("INSERT INTO leads (company,domain,email,notes) VALUES (?,?,?,?)",
                           (lead.company,lead.domain,lead.email,lead.notes))
        conn.commit()
        return dict(conn.execute("SELECT * FROM leads WHERE id=?", (cur.lastrowid,)).fetchone())
    except sqlite3.IntegrityError:
        raise HTTPException(409, "Domain exists")
    finally:
        conn.close()

@app.patch("/api/leads/{lead_id}", response_model=LeadOut)
def update_lead(lead_id: int, lead: LeadUpdate):
    conn = get_db()
    if not conn.execute("SELECT 1 FROM leads WHERE id=?", (lead_id,)).fetchone():
        conn.close(); raise HTTPException(404, "Not found")
    updates = {k: v for k, v in lead.model_dump().items() if v is not None}
    if updates:
        fields = ", ".join(f"{k}=?" for k in updates)
        conn.execute(f"UPDATE leads SET {fields} WHERE id=?", (*updates.values(), lead_id))
        conn.commit()
    row = conn.execute("SELECT * FROM leads WHERE id=?", (lead_id,)).fetchone()
    conn.close()
    return dict(row)

@app.delete("/api/leads/{lead_id}", status_code=204)
def delete_lead(lead_id: int):
    conn = get_db()
    if not conn.execute("SELECT 1 FROM leads WHERE id=?", (lead_id,)).fetchone():
        conn.close(); raise HTTPException(404, "Not found")
    conn.execute("DELETE FROM leads WHERE id=?", (lead_id,))
    conn.commit(); conn.close()

@app.get("/api/stats")
def stats():
    conn = get_db()
    total = conn.execute("SELECT COUNT(*) FROM leads").fetchone()[0]
    conn.close()
    return {"total_leads": total}

# --- Scraper ---
def scrape_email(domain):
    try:
        req = urllib.request.Request(f"https://{domain}", headers={"User-Agent": "Mozilla/5.0"})
        html = urllib.request.urlopen(req, timeout=5).read().decode(errors="ignore")
        emails = list(set(re.findall(r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}", html)))
        return emails[0] if emails else f"contact@{domain}"
    except:
        return f"contact@{domain}"

targets = [
    {"company": "Python.org",        "domain": "python.org"},
    {"company": "Hacker News",        "domain": "news.ycombinator.com"},
    {"company": "Requests Lib",       "domain": "requests.readthedocs.io"},
    {"company": "FastAPI",            "domain": "fastapi.tiangolo.com"},
    {"company": "SQLite",             "domain": "sqlite.org"},
]

print("Scraping targets...")
scraped = []
for t in targets:
    email = scrape_email(t["domain"])
    scraped.append({**t, "email": email})
    print(f"  {t['domain']:40s} -> {email}")

print("\nPushing to API...")
client = TestClient(app)
for lead in scraped:
    r = client.post("/api/leads", json=lead)
    status = "added" if r.status_code == 201 else f"skip ({r.json().get('detail','')})"
    print(f"  {lead['company']:25s} {status}")

# --- Tests ---
new_lead = client.post("/api/leads", json={"company":"TestCo","domain":"testco.pl","email":"a@testco.pl"})
new_id = new_lead.json().get("id")

tests = [
    ("GET all leads",    lambda: client.get("/api/leads"),                                                              200),
    ("GET one lead",     lambda: client.get(f"/api/leads/{new_id}"),                                                    200),
    ("GET stats",        lambda: client.get("/api/stats"),                                                              200),
    ("POST new lead",    lambda: new_lead,                                                                              201),
    ("GET search",       lambda: client.get("/api/leads?search=TestCo"),                                                200),
    ("PATCH lead",       lambda: client.patch(f"/api/leads/{new_id}", json={"company":"TestCo Updated"}),               200),
    ("POST duplicate",   lambda: client.post("/api/leads", json={"company":"X","domain":"testco.pl","email":"b@x.pl"}), 409),
    ("POST bad email",   lambda: client.post("/api/leads", json={"company":"X","domain":"x.pl","email":"notanemail"}),  422),
    ("GET not found",    lambda: client.get("/api/leads/99999"),                                                        404),
    ("DELETE lead",      lambda: client.delete(f"/api/leads/{new_id}"),                                                 204),
    ("GET deleted",      lambda: client.get(f"/api/leads/{new_id}"),                                                    404),
]

print(f"\n{'TEST':<25} {'EXPECT':>6} {'GOT':>6} {'RESULT'}")
print("-" * 50)
passed = 0
for name, fn, expected in tests:
    r = fn()
    ok = r.status_code == expected
    passed += ok
    print(f"{name:<25} {expected:>6} {r.status_code:>6}  {'✓' if ok else '✗ FAIL'}")
print(f"\n{passed}/{len(tests)} passed")

# --- Dashboard ---
leads = client.get("/api/leads").json()
stats_data = client.get("/api/stats").json()

rows = "".join(f"""
  <tr>
    <td>{l['id']}</td>
    <td>{l['company']}</td>
    <td>{l['domain']}</td>
    <td>{l['email']}</td>
    <td>{l.get('notes') or '—'}</td>
  </tr>""" for l in leads)

display(HTML(f"""
<table border="1" cellpadding="8" cellspacing="0"
  style="border-collapse:collapse;width:100%;font-family:monospace;font-size:13px">
  <tr style="background:#222;color:#4f8ef7">
    <th>#</th><th>Company</th><th>Domain</th><th>Email</th><th>Notes</th>
  </tr>
  {rows}
</table>
<p style="font-family:monospace"><b>Total: {stats_data['total_leads']} leads</b></p>
"""))

Scraping targets...
  python.org                               -> contact@python.org
  news.ycombinator.com                     -> hn@ycombinator.com
  requests.readthedocs.io                  -> contact@requests.readthedocs.io
  fastapi.tiangolo.com                     -> contact@fastapi.tiangolo.com
  sqlite.org                               -> contact@sqlite.org

Pushing to API...
  Python.org                added
  Hacker News               added
  Requests Lib              added
  FastAPI                   added
  SQLite                    added

TEST                      EXPECT    GOT RESULT
--------------------------------------------------
GET all leads                200    200  ✓
GET one lead                 200    200  ✓
GET stats                    200    200  ✓
POST new lead                201    201  ✓
GET search                   200    200  ✓
PATCH lead                   200    200  ✓
POST duplicate               409    409  ✓
POST bad email               422    422  ✓


#,Company,Domain,Email,Notes
1,Python.org,python.org,contact@python.org,—
2,Hacker News,news.ycombinator.com,hn@ycombinator.com,—
3,Requests Lib,requests.readthedocs.io,contact@requests.readthedocs.io,—
4,FastAPI,fastapi.tiangolo.com,contact@fastapi.tiangolo.com,—
5,SQLite,sqlite.org,contact@sqlite.org,—
